# 🏰 HTR Medieval French — Fine-tuning sur Google Colab

**Objectif :** Fine-tuner Kraken et TrOCR sur les manuscrits médiévaux français (CREMMA) pour atteindre un CER < 10%.

**Pipeline :**
1. Installation des dépendances
2. Téléchargement des données (CREMMA Médiéval)
3. Préparation du dataset (extraction des lignes depuis ALTO XML)
4. Fine-tuning Kraken (CNN+LSTM)
5. Fine-tuning TrOCR avec LoRA (Vision Transformer)
6. Évaluation et comparaison des résultats

**⚠️ Important :** Activer le GPU dans Colab → Runtime → Change runtime type → T4 GPU

## 1. Installation des dépendances

In [ ]:
# Vérifier qu'on a bien un GPU
!nvidia-smi

In [ ]:
%%capture
# Installation des packages nécessaires
!pip install kraken==5.2.9
!pip install transformers==4.40.0
!pip install peft==0.10.0
!pip install datasets==2.19.0
!pip install editdistance==0.8.1
!pip install Pillow>=10.0
!pip install scikit-learn>=1.5
!pip install tqdm
!pip install accelerate>=0.26.0
!pip install sentencepiece
!pip install lxml

In [ ]:
import os
import json
import random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

# Vérification GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Fixer les seeds pour reproductibilité
def fix_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

fix_seeds(42)

In [ ]:
# 💾 Monter Google Drive pour persister les checkpoints entre sessions
from google.colab import drive
drive.mount('/content/drive')

# Créer le dossier de sauvegarde sur Drive
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/htr-medieval-french/models/trocr-cremma-lora'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f'✅ Google Drive monté — checkpoints sauvegardés dans:\n   {DRIVE_OUTPUT_DIR}')

## 2. Téléchargement des données CREMMA Médiéval

In [ ]:
# Cloner le dépôt CREMMA Médiéval
!git clone https://github.com/HTR-United/cremma-medieval data/cremma
print("\n✅ CREMMA Médiéval cloné avec succès")

In [ ]:
# Explorer la structure
cremma_dir = Path("data/cremma/data")
manuscripts = sorted([d.name for d in cremma_dir.iterdir() if d.is_dir()])
print(f"Nombre de manuscrits : {len(manuscripts)}")
for ms in manuscripts:
    n_xml = len(list((cremma_dir / ms).glob("*.xml")))
    n_jpg = len(list((cremma_dir / ms).glob("*.jpg")))
    # Ne compter que les XML non-chocomufin
    n_xml_clean = len([f for f in (cremma_dir / ms).glob("*.xml") if "chocomufin" not in f.name])
    print(f"  {ms}: {n_jpg} images, {n_xml_clean} XML")

## 3. Extraction des lignes depuis ALTO XML

Les données CREMMA sont au format ALTO XML. Chaque fichier XML contient les coordonnées des lignes de texte et leur transcription. On va extraire chaque ligne comme une image cropée + sa transcription.

In [ ]:
import xml.etree.ElementTree as ET

def extract_lines_from_alto(xml_path, img_path):
    """
    Extrait les lignes d'un fichier ALTO XML.
    Retourne une liste de dicts {'image': PIL.Image, 'text': str, 'line_id': str}
    """
    records = []
    
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError:
        return records
    
    # Détecter le namespace ALTO
    ns = ""
    if root.tag.startswith("{"):
        ns = root.tag.split("}")[0] + "}"
    
    # Charger l'image source
    try:
        page_img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"  [SKIP] Cannot open {img_path}: {e}")
        return records
    
    # Parser les TextLines
    for i, text_line in enumerate(root.iter(f"{ns}TextLine")):
        # Récupérer le texte depuis les éléments String
        parts = []
        for string_el in text_line.iter(f"{ns}String"):
            content = string_el.get("CONTENT", "")
            if content:
                parts.append(content)
        
        text = " ".join(parts).strip()
        if not text or len(text) < 2:
            continue
        
        # Récupérer les coordonnées (HPOS, VPOS, WIDTH, HEIGHT)
        try:
            hpos = float(text_line.get("HPOS", 0))
            vpos = float(text_line.get("VPOS", 0))
            width = float(text_line.get("WIDTH", 0))
            height = float(text_line.get("HEIGHT", 0))
        except (ValueError, TypeError):
            continue
        
        if width <= 0 or height <= 0:
            continue
        
        # Cropper la ligne
        x0 = max(0, int(hpos))
        y0 = max(0, int(vpos))
        x1 = min(page_img.width, int(hpos + width))
        y1 = min(page_img.height, int(vpos + height))
        
        if x1 <= x0 or y1 <= y0:
            continue
        
        line_img = page_img.crop((x0, y0, x1, y1))
        
        line_id = f"{xml_path.stem}_l{i:03d}"
        records.append({
            "image": line_img,
            "text": text,
            "line_id": line_id,
        })
    
    return records

In [ ]:
# Extraire toutes les lignes de CREMMA — version memory-efficient
# On sauvegarde les images directement sur disque au lieu de les garder en RAM

lines_dir = Path("data/lines")
lines_dir.mkdir(parents=True, exist_ok=True)

all_records = []  # Ne contient que les métadonnées (pas les images PIL)
cremma_dir = Path("data/cremma/data")
line_counter = 0

for ms_dir in sorted(cremma_dir.iterdir()):
    if not ms_dir.is_dir():
        continue
    
    ms_name = ms_dir.name
    xml_files = sorted([
        f for f in ms_dir.glob("*.xml")
        if "chocomufin" not in f.name
    ])
    
    ms_count = 0
    for xml_path in xml_files:
        img_path = xml_path.with_suffix(".jpg")
        if not img_path.exists():
            continue
        
        records = extract_lines_from_alto(xml_path, img_path)
        for r in records:
            # Sauvegarder l'image sur disque immédiatement
            line_img_path = lines_dir / f"{line_counter:05d}.png"
            r["image"].save(line_img_path)
            
            # Ne garder que les métadonnées en RAM
            all_records.append({
                "img_path": str(line_img_path),
                "text": r["text"],
                "line_id": r["line_id"],
                "manuscript": ms_name,
            })
            line_counter += 1
        
        ms_count += len(records)
    
    print(f"  {ms_name}: {ms_count} lignes")

print(f"\n✅ Total: {len(all_records)} lignes extraites et sauvées sur disque")
print(f"💾 RAM utilisée: métadonnées seulement (pas d'images en mémoire)")

In [ ]:
# Visualiser quelques exemples
fig, axes = plt.subplots(5, 1, figsize=(14, 10))
sample_indices = random.sample(range(len(all_records)), 5)

for ax, idx in zip(axes, sample_indices):
    record = all_records[idx]
    img = Image.open(record["img_path"])
    ax.imshow(img)
    ax.set_title(f"[{record['manuscript'][:20]}] {record['text'][:80]}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.suptitle("Exemples de lignes extraites", fontsize=12, y=1.02)
plt.show()

## 4. Split Train/Val par manuscrit

**Important :** On split par manuscrit (pas par ligne) pour éviter le data leakage. Si des lignes du même manuscrit sont dans le train et le val, le modèle pourrait mémoriser le style d'écriture au lieu de généraliser.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Split par manuscrit: 85% train, 15% val
manuscripts_list = [r["manuscript"] for r in all_records]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(all_records, groups=manuscripts_list))

train_records = [all_records[i] for i in train_idx]
val_records = [all_records[i] for i in val_idx]

train_ms = set(r["manuscript"] for r in train_records)
val_ms = set(r["manuscript"] for r in val_records)

print(f"Train: {len(train_records)} lignes ({len(train_ms)} manuscrits)")
print(f"Val:   {len(val_records)} lignes ({len(val_ms)} manuscrits)")
print(f"\nManuscrits train: {sorted(train_ms)}")
print(f"Manuscrits val:   {sorted(val_ms)}")

# Vérifier qu'il n'y a pas de fuite
assert len(train_ms & val_ms) == 0, "Data leakage détecté!"
print("\n✅ Pas de data leakage — aucun manuscrit partagé entre train et val")

---
## 5. Fine-tuning Kraken (CNN+LSTM)

Kraken utilise `ketos train` en ligne de commande. On va :
1. Télécharger le modèle pré-entraîné CREMMA
2. Préparer les données au format attendu par ketos
3. Lancer le fine-tuning

In [ ]:
# Télécharger le modèle Kraken CREMMA pré-entraîné
!mkdir -p models

# Le modèle est disponible directement via kraken
!kraken get 10.5281/zenodo.7234166
print("\n✅ Modèle CREMMA téléchargé")

# Trouver le chemin du modèle téléchargé
import glob
model_files = glob.glob(os.path.expanduser("~/.config/kraken/*.mlmodel")) + \
              glob.glob(os.path.expanduser("~/.local/share/kraken/*.mlmodel")) + \
              glob.glob("*.mlmodel")
print(f"Modèles trouvés: {model_files}")

In [ ]:
# Préparer les données pour ketos (format binaire avec paires image/texte)
# ketos attend des fichiers .png (ligne) + .gt.txt (transcription)

kraken_train_dir = Path("data/kraken_train")
kraken_val_dir = Path("data/kraken_val")
kraken_train_dir.mkdir(parents=True, exist_ok=True)
kraken_val_dir.mkdir(parents=True, exist_ok=True)

def save_kraken_format(records, output_dir):
    """Copie les lignes au format attendu par ketos (image + .gt.txt)."""
    manifest = []
    for i, record in enumerate(tqdm(records, desc=f"Saving to {output_dir.name}")):
        img_path = output_dir / f"line_{i:05d}.png"
        txt_path = output_dir / f"line_{i:05d}.gt.txt"
        
        # Copier/convertir l'image en niveaux de gris
        img = Image.open(record["img_path"]).convert("L")
        img.save(img_path)
        
        # Sauvegarder la transcription
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(record["text"])
        
        manifest.append(str(img_path))
    
    return manifest

train_manifest = save_kraken_format(train_records, kraken_train_dir)
val_manifest = save_kraken_format(val_records, kraken_val_dir)

print(f"\n✅ Train: {len(train_manifest)} fichiers dans {kraken_train_dir}")
print(f"✅ Val:   {len(val_manifest)} fichiers dans {kraken_val_dir}")

In [ ]:
# Créer les fichiers manifest pour ketos
with open("train_manifest.txt", "w") as f:
    for path in train_manifest:
        f.write(path + "\n")

with open("val_manifest.txt", "w") as f:
    for path in val_manifest:
        f.write(path + "\n")

print("Manifests créés")

In [ ]:
%%time
# Fine-tuning Kraken
# --resize: new → entraîner un nouveau modèle à partir du codec
# -f binary: format des données (image + .gt.txt)
# --augment: augmentation de données
# -d cuda:0: utiliser le GPU

# Option A: Entraîner from scratch sur CREMMA
!ketos train \
    -f binary \
    -d cuda:0 \
    --resize add \
    --augment \
    --workers 2 \
    --lag 10 \
    --min-epochs 5 \
    --epochs 50 \
    -o models/kraken_cremma_finetuned \
    --training-files train_manifest.txt \
    --evaluation-files val_manifest.txt

In [ ]:
# Trouver le meilleur modèle Kraken sauvegardé
kraken_models = sorted(glob.glob("models/kraken_cremma_finetuned*.mlmodel"))
if kraken_models:
    best_kraken = kraken_models[-1]  # le dernier est le best
    print(f"✅ Meilleur modèle Kraken: {best_kraken}")
else:
    print("⚠️ Aucun modèle Kraken trouvé — vérifier l'entraînement")

---
## 6. Fine-tuning TrOCR avec LoRA

TrOCR = Vision Encoder-Decoder (ViT + GPT-2 decoder). On utilise LoRA pour ne fine-tuner que ~1% des paramètres.

**Avantages :**
- Beaucoup moins de VRAM nécessaire
- Entraînement plus rapide
- Moins de risque d'overfitting sur un petit dataset

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import editdistance

print("✅ Imports OK")

In [ ]:
# Charger le processor et le modèle de base
MODEL_NAME = "microsoft/trocr-base-handwritten"

print(f"Chargement de {MODEL_NAME}...")
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

print(f"Paramètres totaux: {model.num_parameters():,}")
print(f"Taille du modèle: ~{model.num_parameters() * 4 / 1e9:.1f} GB (FP32)")

In [ ]:
# Appliquer LoRA
LORA_R = 8  # Rang LoRA (essayer 8 puis 16 si CER stagne)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 4,  # = 32
    target_modules=["query", "value"],  # Attention layers
    lora_dropout=0.1,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Devrait montrer ~1% de paramètres entraînables

In [ ]:
# Configurer le decoder
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Déplacer sur GPU
model = model.to(device)
print(f"Modèle sur {device}")

In [ ]:
# Dataset LAZY — ne charge les images qu'au moment du batch (économise la RAM)
MAX_LENGTH = 128

class LazyHTRDataset(torch.utils.data.Dataset):
    """
    Dataset qui charge les images depuis le disque à la volée.
    Évite de garder tous les pixel_values en RAM.
    """
    def __init__(self, records, processor, max_length=MAX_LENGTH):
        self.processor = processor
        self.max_length = max_length
        self.img_paths = [r["img_path"] for r in records]
        self.texts = [r["text"] for r in records]
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # Charger l'image à la volée (pas stockée en RAM)
        img = Image.open(self.img_paths[idx]).convert("RGB")
        pixel_values = self.processor(img, return_tensors="pt").pixel_values[0]
        
        # Tokenizer le texte
        labels = self.processor.tokenizer(
            self.texts[idx],
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt",
        ).input_ids[0]
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        return {"pixel_values": pixel_values, "labels": labels}

print("Création du dataset train...")
train_dataset = LazyHTRDataset(train_records, processor)

print("Création du dataset val...")
val_dataset = LazyHTRDataset(val_records, processor)

# Libérer la RAM
import gc
del all_records
gc.collect()

print(f"\n✅ Train: {len(train_dataset)} samples")
print(f"✅ Val:   {len(val_dataset)} samples")
print(f"💾 Images chargées à la volée — RAM libérée")

In [ ]:
# Fonction pour calculer le CER
def compute_cer(predictions, references):
    """Character Error Rate = edit_distance / total_chars"""
    total_errors = sum(
        editdistance.eval(p, r)
        for p, r in zip(predictions, references)
    )
    total_chars = sum(len(r) for r in references)
    if total_chars == 0:
        return 0.0
    return total_errors / total_chars

def compute_metrics(pred):
    """Métriques pour le Trainer."""
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    
    # Remplacer -100 par pad_token_id pour le décodage
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    
    cer = compute_cer(pred_str, label_str)
    return {"cer": cer}

In [ ]:
# Configuration de l'entraînement
EPOCHS = 10  # Réduit de 30 à 10 pour tenir dans la session Colab
BATCH_SIZE = 8  # Augmenté de 4 à 8 pour accélérer (si OOM, remettre à 4)
LEARNING_RATE = 5e-5
OUTPUT_DIR = DRIVE_OUTPUT_DIR  # Sauvegarde sur Google Drive pour persister entre sessions

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    fp16=True,  # Mixed precision pour T4
    seed=42,
    logging_steps=25,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    save_total_limit=3,  # Garder seulement les 3 meilleurs checkpoints
    dataloader_num_workers=0,  # 0 pour éviter le fork de RAM
)

print("Training config:")
print(f"  Epochs:         {EPOCHS}")
print(f"  Batch size:     {BATCH_SIZE}")
print(f"  Learning rate:  {LEARNING_RATE}")
print(f"  LoRA rank:      {LORA_R}")
print(f"  Output:         {OUTPUT_DIR}")

In [ ]:
# Créer le Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("✅ Trainer prêt")

In [ ]:
%%time
# 🚀 LANCER LE FINE-TUNING (reprend automatiquement si un checkpoint existe)
import glob

print("="*60)
print("🚀 Début du fine-tuning TrOCR + LoRA")
print("="*60)

# Chercher un checkpoint existant sur Google Drive
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
if checkpoints:
    print(f"📂 Checkpoint trouvé: {checkpoints[-1]}")
    print(f"   → Reprise de l'entraînement...")
    train_result = trainer.train(resume_from_checkpoint=checkpoints[-1])
else:
    print("🆕 Aucun checkpoint — entraînement from scratch")
    train_result = trainer.train()

print("\n" + "="*60)
print("✅ Fine-tuning terminé!")
print(f"   Train loss: {train_result.training_loss:.4f}")
print("="*60)

In [ ]:
# Sauvegarder le modèle final
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"✅ Modèle sauvegardé dans {OUTPUT_DIR}")

In [ ]:
# Évaluation finale
eval_results = trainer.evaluate()

print("\n" + "="*60)
print("📊 RÉSULTATS FINAUX")
print("="*60)
print(f"CER validation: {eval_results['eval_cer']:.4f} ({eval_results['eval_cer']*100:.1f}%)")
print(f"Loss validation: {eval_results['eval_loss']:.4f}")
print("="*60)

if eval_results['eval_cer'] < 0.10:
    print("\n🎉 Objectif atteint! CER < 10%")
else:
    print(f"\n📈 CER = {eval_results['eval_cer']*100:.1f}% — essayer:")
    print("   • Plus d'epochs")
    print("   • LoRA r=16 au lieu de r=8")
    print("   • Ajouter les données CATMuS")
    print("   • Post-correction LLM (Mistral/GPT-4)")

## 7. Évaluation qualitative

In [ ]:
# Inférence sur quelques exemples de validation
model.eval()

n_examples = 10
sample_indices = random.sample(range(len(val_records)), min(n_examples, len(val_records)))

print(f"{'='*80}")
print(f"{'Idx':<5} {'Référence':<40} {'Prédiction':<40}")
print(f"{'='*80}")

preds_sample = []
refs_sample = []

for idx in sample_indices:
    record = val_records[idx]
    img = Image.open(record["img_path"]).convert("RGB")
    
    # Inférence
    pixel_values = processor(img, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_new_tokens=128)
    
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ref_text = record["text"]
    
    preds_sample.append(pred_text)
    refs_sample.append(ref_text)
    
    # CER par ligne
    line_cer = editdistance.eval(pred_text, ref_text) / max(len(ref_text), 1)
    marker = "✓" if line_cer < 0.10 else "✗"
    
    print(f"{marker} REF: {ref_text[:70]}")
    print(f"  PRED: {pred_text[:70]}")
    print(f"  CER:  {line_cer:.1%}")
    print()

sample_cer = compute_cer(preds_sample, refs_sample)
print(f"\nCER sur cet échantillon: {sample_cer:.1%}")

In [ ]:
# Visualisation avec images
fig, axes = plt.subplots(5, 1, figsize=(14, 12))

for i, (ax, idx) in enumerate(zip(axes, sample_indices[:5])):
    record = val_records[idx]
    img = Image.open(record["img_path"])
    ax.imshow(img)
    
    ref = refs_sample[i][:60]
    pred = preds_sample[i][:60]
    line_cer = editdistance.eval(preds_sample[i], refs_sample[i]) / max(len(refs_sample[i]), 1)
    
    color = "green" if line_cer < 0.10 else "orange" if line_cer < 0.30 else "red"
    ax.set_title(
        f"REF: {ref}\nPRED: {pred}  (CER: {line_cer:.0%})",
        fontsize=9, color=color, loc="left"
    )
    ax.axis("off")

plt.tight_layout()
plt.savefig("trocr_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé dans trocr_predictions.png")

## 8. Courbes d'entraînement

In [ ]:
# Tracer les courbes de loss et CER
history = trainer.state.log_history

train_losses = [(h["step"], h["loss"]) for h in history if "loss" in h and "eval_loss" not in h]
eval_losses = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]
eval_cers = [(h["step"], h["eval_cer"]) for h in history if "eval_cer" in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
if train_losses:
    ax1.plot(*zip(*train_losses), label="Train loss", alpha=0.7)
if eval_losses:
    ax1.plot(*zip(*eval_losses), label="Val loss", linewidth=2)
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend()
ax1.grid(alpha=0.3)

# CER
if eval_cers:
    steps, cers = zip(*eval_cers)
    ax2.plot(steps, [c*100 for c in cers], 'o-', color="#e74c3c", linewidth=2)
    ax2.axhline(10, color="green", linestyle="--", label="Target (10%)")
    ax2.set_xlabel("Step")
    ax2.set_ylabel("CER (%)")
    ax2.set_title("Character Error Rate")
    ax2.legend()
    ax2.grid(alpha=0.3)

plt.suptitle(f"TrOCR + LoRA (r={LORA_R}) — Fine-tuning sur CREMMA", fontsize=13)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## 9. Sauvegarder et télécharger le modèle

In [ ]:
# Compresser le modèle pour le télécharger
import shutil

# Sauvegarder les résultats
results = {
    "model": "trocr-base-handwritten + LoRA",
    "lora_r": LORA_R,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "train_lines": len(train_records),
    "val_lines": len(val_records),
    "final_cer": eval_results['eval_cer'],
    "final_loss": eval_results['eval_loss'],
    "train_manuscripts": sorted(list(train_ms)),
    "val_manuscripts": sorted(list(val_ms)),
}

with open(f"{OUTPUT_DIR}/training_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

In [ ]:
# Compresser pour téléchargement
shutil.make_archive("trocr_cremma_finetuned", "zip", OUTPUT_DIR)
print(f"\n📦 Modèle compressé: trocr_cremma_finetuned.zip")
print("   Télécharger depuis le panneau fichiers à gauche")

# Si les modèles Kraken existent aussi
if kraken_models:
    print(f"\n📦 Modèle Kraken: {best_kraken}")

In [ ]:
# Optionnel: monter Google Drive et y copier le modèle
from google.colab import drive
drive.mount('/content/drive')

# Copier le modèle sur Drive
drive_output = Path("/content/drive/MyDrive/htr-medieval-french/models")
drive_output.mkdir(parents=True, exist_ok=True)

shutil.copy("trocr_cremma_finetuned.zip", drive_output / "trocr_cremma_finetuned.zip")
print(f"✅ Modèle copié sur Google Drive: {drive_output}")

# Copier aussi les courbes et résultats
for f in ["training_curves.png", "trocr_predictions.png"]:
    if Path(f).exists():
        shutil.copy(f, drive_output / f)

---
## 10. Bonus: Inférence sur vos propres images

Utilisez cette cellule pour tester le modèle fine-tuné sur de nouvelles images manuscrites.

In [ ]:
def transcribe_line(image_path_or_pil, model, processor, device="cuda"):
    """
    Transcrire une image de ligne avec le modèle fine-tuné.
    
    Args:
        image_path_or_pil: Chemin vers l'image ou objet PIL.Image
        model: Modèle TrOCR fine-tuné
        processor: TrOCRProcessor
        device: 'cuda' ou 'cpu'
    
    Returns:
        Transcription str
    """
    if isinstance(image_path_or_pil, (str, Path)):
        img = Image.open(image_path_or_pil).convert("RGB")
    else:
        img = image_path_or_pil.convert("RGB")
    
    pixel_values = processor(img, return_tensors="pt").pixel_values.to(device)
    
    model.eval()
    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_new_tokens=128)
    
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text

# Test sur un exemple
test_record = val_records[0]
pred = transcribe_line(test_record["image"], model, processor, device)
print(f"REF:  {test_record['text']}")
print(f"PRED: {pred}")

In [ ]:
# Pour charger le modèle plus tard (depuis le zip sauvegardé):
# processor = TrOCRProcessor.from_pretrained("models/trocr-cremma-lora")
# model = VisionEncoderDecoderModel.from_pretrained("models/trocr-cremma-lora")
# model = model.to("cuda")

print("\n" + "="*60)
print("🏁 NOTEBOOK TERMINÉ")
print("="*60)
print(f"\nRésumé:")
print(f"  • Données: CREMMA Médiéval ({len(all_records)} lignes)")
print(f"  • Modèle: TrOCR + LoRA (r={LORA_R})")
print(f"  • CER final: {eval_results['eval_cer']*100:.1f}%")
print(f"  • Modèle sauvegardé: {OUTPUT_DIR}")
print(f"\nPour améliorer le CER:")
print(f"  1. Augmenter LoRA r (16 ou 32)")
print(f"  2. Ajouter CATMuS (plus de données)")
print(f"  3. Post-correction LLM (Mistral/GPT-4o)")
print(f"  4. Entraîner plus longtemps (50+ epochs)")